In [1]:
import pandas as pd

df = pd.read_csv("../data/Womens Clothing E-Commerce Reviews.csv")

print(df.shape)

(23486, 11)


In [2]:
#Select low-rated reviews

low_rated = df[df["Rating"] <= 3].copy()

print("Low-rated reviews:", len(low_rated))

Low-rated reviews: 5278


In [3]:
#Select the 200 reviews

sample_200 = low_rated.sample(
    n=200,
    random_state=42
).copy()

print("Sample size:", len(sample_200))

Sample size: 200


In [4]:
#Check missing review text

missing_text = sample_200["Review Text"].isna().sum()

print("Reviews with missing text:", missing_text)

Reviews with missing text: 7


In [5]:
#Remove reviews with missing text

sample_200 = sample_200[
    sample_200["Review Text"].notna()
].copy()

print("Reviews with text:", len(sample_200))


Reviews with text: 193


In [6]:
labels = [
    "size/fit",
    "fabric/quality",
    "looks different from photo",
    "comfort",
    "price",
    "other"
]

print("Labels:", labels)

Labels: ['size/fit', 'fabric/quality', 'looks different from photo', 'comfort', 'price', 'other']


In [7]:
#Create the LLM prompt

prompt_template = """
You are classifying customer reviews for a women's clothing retailer.

Choose exactly ONE main issue from these labels:
1. size/fit
2. fabric/quality
3. looks different from photo
4. comfort
5. price
6. other

Also identify whether the review says the item runs:
- small
- large
- neither

Return ONLY this format:
issue|fit

Do not add explanations.

Review:
{review}
"""

print(prompt_template)


You are classifying customer reviews for a women's clothing retailer.

Choose exactly ONE main issue from these labels:
1. size/fit
2. fabric/quality
3. looks different from photo
4. comfort
5. price
6. other

Also identify whether the review says the item runs:
- small
- large
- neither

Return ONLY this format:
issue|fit

Do not add explanations.

Review:
{review}



In [13]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv("../.env")

groq_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=groq_key)

print("Groq client ready")

Groq client ready


In [11]:
#Test the LLM with one review

test_review = sample_200.iloc[0]["Review Text"]

prompt = prompt_template.format(review=test_review)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

print("Groq response:")
print(response.choices[0].message.content.strip())

Groq response:
size/fit|small


In [14]:
import time

results = []

for i, row in sample_200.iterrows():
    review = row["Review Text"]
    prompt = prompt_template.format(review=review)

    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        llm_output = response.choices[0].message.content.strip()

        results.append({
            "Review ID": row["Unnamed: 0"],
            "Clothing ID": row["Clothing ID"],
            "Department Name": row["Department Name"],
            "Class Name": row["Class Name"],
            "Rating": row["Rating"],
            "Review Text": review,
            "LLM_Output": llm_output
        })

        # Save progress after every successful review
        pd.DataFrame(results).to_csv(
            "../outputs/llm_progress.csv",
            index=False
        )

        print(f"Processed {len(results)}/{len(sample_200)}")

    except Exception as e:
        print(f"Stopped at {len(results)}/{len(sample_200)}")
        print(type(e).__name__, ":", e)
        break

results_df = pd.DataFrame(results)

print("Successful results:", len(results_df))

Processed 1/193
Processed 2/193
Processed 3/193
Processed 4/193
Processed 5/193
Processed 6/193
Processed 7/193
Processed 8/193
Processed 9/193
Processed 10/193
Processed 11/193
Processed 12/193
Processed 13/193
Processed 14/193
Processed 15/193
Processed 16/193
Processed 17/193
Processed 18/193
Processed 19/193
Processed 20/193
Processed 21/193
Processed 22/193
Processed 23/193
Processed 24/193
Processed 25/193
Processed 26/193
Processed 27/193
Processed 28/193
Processed 29/193
Processed 30/193
Processed 31/193
Processed 32/193
Processed 33/193
Processed 34/193
Processed 35/193
Processed 36/193
Processed 37/193
Processed 38/193
Processed 39/193
Processed 40/193
Processed 41/193
Processed 42/193
Processed 43/193
Processed 44/193
Processed 45/193
Processed 46/193
Processed 47/193
Processed 48/193
Processed 49/193
Processed 50/193
Processed 51/193
Processed 52/193
Processed 53/193
Processed 54/193
Processed 55/193
Processed 56/193
Processed 57/193
Processed 58/193
Processed 59/193
Proces

In [27]:
import time
import pandas as pd

# Load saved progress
results_df = pd.read_csv("../outputs/llm_progress.csv")
results = results_df.to_dict("records")

print(f"Already completed: {len(results)}/{len(sample_200)}")
print(f"Continuing from: {len(results) + 1}")

i = len(results)

while i < len(sample_200):

    row = sample_200.iloc[i]
    review = row["Review Text"]
    prompt = prompt_template.format(review=review)

    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        llm_output = response.choices[0].message.content.strip()

        results.append({
            "Review ID": row["Unnamed: 0"],
            "Clothing ID": row["Clothing ID"],
            "Department Name": row["Department Name"],
            "Class Name": row["Class Name"],
            "Rating": row["Rating"],
            "Review Text": review,
            "LLM_Output": llm_output
        })

        i += 1

        # Save after every successful review
        pd.DataFrame(results).to_csv(
            "../outputs/llm_progress.csv",
            index=False
        )

        print(f"Processed {i}/{len(sample_200)}")

        # Slow down requests
        time.sleep(40)

    except Exception as e:

        if "429" in str(e):
            print("Rate limit reached. Waiting 180 seconds...")
            time.sleep(180)
            continue

        print(f"Stopped at {i}/{len(sample_200)}")
        print(type(e).__name__, ":", e)
        break

results_df = pd.DataFrame(results)

print("Successful results:", len(results_df))

Already completed: 192/193
Continuing from: 193
Processed 193/193
Successful results: 193


In [28]:
# Save final LLM results
results_df.to_csv(
    "../outputs/llm_tagged_193_reviews.csv",
    index=False
)

print("Saved: outputs/llm_tagged_193_reviews.csv")
print("Rows:", len(results_df))

Saved: outputs/llm_tagged_193_reviews.csv
Rows: 193


In [29]:
# Parse LLM output into Issue and Fit columns

def parse_output(output):
    try:
        issue, fit = output.split("|")
        return issue.strip(), fit.strip()
    except:
        return "parse_error", "parse_error"

results_df[["LLM_Issue", "LLM_Fit"]] = (
    results_df["LLM_Output"]
    .apply(parse_output)
    .apply(pd.Series)
)

print("Parsed results:", len(results_df))

print("\nIssue counts:")
print(results_df["LLM_Issue"].value_counts())

print("\nFit counts:")
print(results_df["LLM_Fit"].value_counts())

Parsed results: 193

Issue counts:
LLM_Issue
size/fit                      135
fabric/quality                 30
looks different from photo     11
other                           7
parse_error                     4
price                           3
comfort                         2
2                               1
Name: count, dtype: int64

Fit counts:
LLM_Fit
neither        71
large          67
small          51
parse_error     4
Name: count, dtype: int64


In [30]:
# Inspect outputs that were not parsed correctly

problem_rows = results_df[
    (results_df["LLM_Issue"] == "parse_error") |
    (results_df["LLM_Fit"] == "parse_error") |
    (~results_df["LLM_Issue"].isin([
        "size/fit",
        "fabric/quality",
        "looks different from photo",
        "comfort",
        "price",
        "other"
    ]))
]

print("Problematic rows:", len(problem_rows))

problem_rows[
    ["Review ID", "Rating", "Review Text", "LLM_Output", "LLM_Issue", "LLM_Fit"]
]

Problematic rows: 5


,Review ID,Rating,Review Text,LLM_Output,LLM_Issue,LLM_Fit
106,23334,3,I ordered the dress online and found it to be ...,NaN,parse_error,parse_error
129,22126,1,Super itchy! had to return.,2|neither,2,neither
176,22824,2,"1. i'm 5'6"" tall with a short waist and long l...",looks different from photo|neither\nsize/fit|n...,parse_error,parse_error
184,1958,3,Facing the mirror--near perfection. high waist...,NaN,parse_error,parse_error
190,10197,1,Large and the material is too bulky. i returne...,NaN,parse_error,parse_error


In [31]:
# Fix the 5 problematic LLM outputs based on the review text

# Review ID 22126: "Super itchy! had to return."
# Main issue = comfort
results_df.loc[
    results_df["Review ID"] == 22126,
    ["LLM_Issue", "LLM_Fit"]
] = ["comfort", "neither"]

# Review ID 22824: review discusses appearance compared with expectation
results_df.loc[
    results_df["Review ID"] == 22824,
    ["LLM_Issue", "LLM_Fit"]
] = ["looks different from photo", "neither"]

# Review ID 23334: keep as other because the LLM returned no usable output
results_df.loc[
    results_df["Review ID"] == 23334,
    ["LLM_Issue", "LLM_Fit"]
] = ["other", "neither"]

# Review ID 1958: keep as other because the LLM returned no usable output
results_df.loc[
    results_df["Review ID"] == 1958,
    ["LLM_Issue", "LLM_Fit"]
] = ["other", "neither"]

# Review ID 10197: mentions "Large"
results_df.loc[
    results_df["Review ID"] == 10197,
    ["LLM_Issue", "LLM_Fit"]
] = ["size/fit", "large"]

print("Remaining problematic rows:")

valid_issues = [
    "size/fit",
    "fabric/quality",
    "looks different from photo",
    "comfort",
    "price",
    "other"
]

print(
    results_df[
        ~results_df["LLM_Issue"].isin(valid_issues)
    ][["Review ID", "LLM_Output", "LLM_Issue", "LLM_Fit"]]
)

Remaining problematic rows:
Empty DataFrame
Columns: [Review ID, LLM_Output, LLM_Issue, LLM_Fit]
Index: []


In [32]:
# Save final Part C results with parsed labels

results_df.to_csv(
    "../outputs/llm_tagged_193_reviews.csv",
    index=False
)

print("Saved: outputs/llm_tagged_193_reviews.csv")
print("Rows:", len(results_df))
print("Columns:", list(results_df.columns))

Saved: outputs/llm_tagged_193_reviews.csv
Rows: 193
Columns: ['Review ID', 'Clothing ID', 'Department Name', 'Class Name', 'Rating', 'Review Text', 'LLM_Output', 'LLM_Issue', 'LLM_Fit']


In [33]:
# Select 30 reviews for manual validation

manual_check = results_df.sample(
    n=30,
    random_state=42
).copy()

print("Manual check sample:", len(manual_check))

manual_check[
    [
        "Review ID",
        "Rating",
        "Department Name",
        "Review Text",
        "LLM_Issue",
        "LLM_Fit"
    ]
]

Manual check sample: 30


,Review ID,Rating,Department Name,Review Text,LLM_Issue,LLM_Fit
45,16868,3,Dresses,This dress was ok - i thought the fabric was s...,size/fit,small
137,18928,3,Tops,The front and back of this square sweater are ...,size/fit,small
76,10143,3,Tops,"I like what they were going for here, and the ...",size/fit,small
144,4469,3,Tops,I was looking forward to this cute top to get ...,size/fit,large
113,16633,1,Tops,"I was really excited about this sweater, but t...",size/fit,large
152,19248,3,Bottoms,I love the look and feel of these shorts - typ...,size/fit,large
15,13510,2,Tops,I was so excited to order this sweater. i thou...,fabric/quality,neither
18,69,3,Dresses,"I really wanted this to work. alas, it had a s...",size/fit,neither
16,10440,3,Dresses,Very cute and would transition well season to ...,size/fit,neither
9,1450,3,Dresses,The design of this dress is darling and so uni...,size/fit,large


In [34]:
# Create manual validation table

manual_check = manual_check[
    [
        "Review ID",
        "Rating",
        "Department Name",
        "Review Text",
        "LLM_Issue",
        "LLM_Fit"
    ]
].copy()

# Add columns for your manual labels
manual_check["Manual_Issue"] = ""
manual_check["Manual_Fit"] = ""

# Show full review text
pd.set_option("display.max_colwidth", None)

manual_check[
    [
        "Review ID",
        "Review Text",
        "LLM_Issue",
        "LLM_Fit",
        "Manual_Issue",
        "Manual_Fit"
    ]
]

,Review ID,Review Text,LLM_Issue,LLM_Fit,Manual_Issue,Manual_Fit
45,16868,"This dress was ok - i thought the fabric was soft, but not my favorite. it's a pull on dress that would be easy to dress up or down. but, i have bigger hips and this dress was really not flattering on my - would be better on a different body type.",size/fit,small,,
137,18928,"The front and back of this square sweater are the same, and the pattern of the stitching makes it look like it's on backwards.the lace would be nice, but it's itchy. and the sweater is cropped, so i think the layered lace look is odd because the whole thing is so short on the body. the 3/4 sleeves make it look small and fitted, but the body shape is bulky. just couldn't get it to work.",size/fit,small,,
76,10143,"I like what they were going for here, and the fabric. but it falls short in that i can't wear it to work---the back ""keyhole"" is massive so i can't wear my normal bras, and the very wide neck also shows bra straps, so unless i want to go with a strapless bra, it's not useful for me. i want to feel loose and soft and summer comfy and my strapless bras are anything but that. i'd go without, but that is a no-no where i work. i may try getting a size smaller. i wear a med. and i got a small and mayb",size/fit,small,,
144,4469,"I was looking forward to this cute top to get me through the winter, but the combination of the very think fabric and boxy shape did not work on my 5'2"", 130 lb, 34d figure. the one i received also had an open unfinished seam beneath the zipper. i imagine this would be a great top for someone taller.",size/fit,large,,
113,16633,"I was really excited about this sweater, but the fit was terrible. the sleeves were the right size, but the body of the sweater was a tent - absolutely no shape. they must have clipped the sweater on the model. for reference, i'm 5'4 and 120 pounds and ordered a size small.",size/fit,large,,
152,19248,"I love the look and feel of these shorts - typically my issue with any sort of cotton or linen shorts these days is that they're practically made of nothing, but these are comfortable and made of decent fabric. however, they run very large. being 100 lbs and about 5', i typically buy size 0 or xs, but these looked a bit ridiculous on me, with how big they are. i tried shrinking them in the wash, since they're 100% cotton, and that helped some, along with getting rid of the creases from shipping",size/fit,large,,
15,13510,"I was so excited to order this sweater. i thought that the moss color would make it a great basic for work or casual days. ordering it on sale made it all the better. however, when it came i was so sad to see how muted the colors were, with little definition. the sweater itself lacks definition as well, just shapeless and hangy. i also did not expect the angora like hair on the sweater which bothered by eyes as soon as i tried it on.",fabric/quality,neither,,
18,69,"I really wanted this to work. alas, it had a strange fit for me. the straps would not stay up, and it had a weird fit under the breast. it worked standing up, but the minute i sat down it fell off my shoulders. the fabric was beautiful! and i loved that it had pockets.",size/fit,neither,,
16,10440,Very cute and would transition well season to season with different boots/booties. didn't lay well on chest for me. returned it.,size/fit,neither,,
9,1450,"The design of this dress is darling and so unique, and it's very well made. but i agree it's very boxy and overwhelming for a petite frame. i'm 5'3"" and ordered the 2, and it was like wearing a hospital gown. if a petite were available i'd have tried that, but this one's going back. shame, too, because it's adorable. the gap in the neckline could easily be stitched or sewn shut, but that's an inconvenience when this dress is already priced so high.",size/fit,large,,


In [35]:
# Fill manual labels based on review text

manual_labels = {
    16868: ("size/fit", "neither"),
    18928: ("comfort", "neither"),
    10143: ("size/fit", "small"),
    4469: ("size/fit", "neither"),
    16633: ("size/fit", "large"),
    19248: ("size/fit", "large"),
    13510: ("fabric/quality", "neither"),
    69: ("size/fit", "neither"),
    10440: ("size/fit", "neither"),
    1450: ("size/fit", "large"),
    3035: ("fabric/quality", "neither"),
    11415: ("size/fit", "small"),
    19115: ("size/fit", "neither"),
    8758: ("size/fit", "small"),
    15599: ("size/fit", "large"),
    16021: ("size/fit", "large"),
    16189: ("looks different from photo", "large"),
    14365: ("size/fit", "large"),
    18880: ("size/fit", "large"),
    3894: ("looks different from photo", "neither"),
    7604: ("fabric/quality", "large"),
    14010: ("fabric/quality", "neither"),
    5622: ("fabric/quality", "neither"),
    17067: ("fabric/quality", "neither"),
    16807: ("size/fit", "large"),
    7290: ("size/fit", "neither"),
    9539: ("fabric/quality", "neither"),
    11342: ("other", "neither"),
    16625: ("fabric/quality", "neither"),
    503: ("comfort", "neither")
}

for review_id, (issue, fit) in manual_labels.items():
    manual_check.loc[
        manual_check["Review ID"] == review_id,
        ["Manual_Issue", "Manual_Fit"]
    ] = [issue, fit]

print("Manual labels completed:", 
      manual_check["Manual_Issue"].notna().sum())

print("\nManual issue labels:")
print(manual_check["Manual_Issue"].value_counts())

print("\nManual fit labels:")
print(manual_check["Manual_Fit"].value_counts())

Manual labels completed: 30

Manual issue labels:
Manual_Issue
size/fit                      17
fabric/quality                 8
comfort                        2
looks different from photo     2
other                          1
Name: count, dtype: int64

Manual fit labels:
Manual_Fit
neither    17
large      10
small       3
Name: count, dtype: int64


In [36]:
# Calculate manual validation accuracy

manual_check["Issue_Correct"] = (
    manual_check["LLM_Issue"] == manual_check["Manual_Issue"]
)

manual_check["Fit_Correct"] = (
    manual_check["LLM_Fit"] == manual_check["Manual_Fit"]
)

manual_check["Exact_Correct"] = (
    manual_check["Issue_Correct"] &
    manual_check["Fit_Correct"]
)

issue_accuracy = manual_check["Issue_Correct"].mean() * 100
fit_accuracy = manual_check["Fit_Correct"].mean() * 100
exact_accuracy = manual_check["Exact_Correct"].mean() * 100

print(f"Issue accuracy: {issue_accuracy:.2f}%")
print(f"Fit accuracy: {fit_accuracy:.2f}%")
print(f"Exact accuracy: {exact_accuracy:.2f}%")

manual_check.to_csv(
    "../outputs/manual_30_check.csv",
    index=False
)

print("Saved: outputs/manual_30_check.csv")

Issue accuracy: 96.67%
Fit accuracy: 90.00%
Exact accuracy: 90.00%
Saved: outputs/manual_30_check.csv


In [37]:
# Issue mix by department

issue_mix = pd.crosstab(
    results_df["Department Name"],
    results_df["LLM_Issue"]
)

issue_mix = issue_mix.reindex(
    columns=labels,
    fill_value=0
)

issue_mix["Total"] = issue_mix.sum(axis=1)

print("Issue mix by department:")
display(issue_mix)

issue_mix.to_csv(
    "../outputs/issue_mix_by_department.csv"
)

print("Saved: outputs/issue_mix_by_department.csv")

Issue mix by department:


LLM_Issue,size/fit,fabric/quality,looks different from photo,comfort,price,other,Total
Department Name,,,,,,,
Bottoms,20,4,2,0,0,2,28
Dresses,49,10,3,0,0,3,65
Intimate,9,0,1,1,0,1,12
Jackets,4,2,0,0,1,0,7
Tops,53,14,6,2,1,3,79
Trend,1,0,0,0,1,0,2


Saved: outputs/issue_mix_by_department.csv
